<h3>Table 1 — Daily Model Diagnostics</h3>

<div style="overflow-x:auto;">

| column | dtype | description |
|---|---|---|
| train_day | date | Training day |
| test_day | date | Evaluation day |
| symbol | string | Stock symbol |
| target | string | Target column (price measure + forecast horizon), e.g. `T_MidPrice_LogReturn_1s` |
| run_id | int | Identifier for features / preprocessing / model |
| n_train | int | Number of training observations |
| n_test | int | Number of test observations |
| mse | float | Model mean squared error |
| mse_bench | float | Benchmark (zero-return) mean squared error = `mean(y_test²)` |
| mse_ratio | float | `mse_bench / mse` |
| mae | float | Mean absolute error |
| r2_oos | float | Out-of-sample R² = `1 - mse / mse_bench` |

</div>

Planned (not yet computed): `directional_accuracy`, `directional_accuracy_bench` (= 0.5 for the zero-return benchmark), `residual_mean`, `residual_std`.

<hr>

<h3>Table 2 — Run-Specific Identifiers</h3>

<div style="overflow-x:auto;">

| run_id | model_version | feature_set_id | normalization_id |
|---|---|---|---|
| int | string | string | string |

</div>

<hr>

</div>


Add Table to store DM-related outputs. Something like:
Symbol, Trading Day, reg_model_ID, XGBoost_model_ID, LSTM_model_ID, d_reg_XBoost, d_reg_LSTM, ...

In [ ]:
import os, warnings, sys
from pathlib import Path
import time
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression

sys.path.append(os.path.dirname(os.getcwd()))
import importlib
import utils.data_preprocessing as du
import utils.model_utils as mu

importlib.reload(du)
importlib.reload(mu)

In [3]:
SAVE_TICK_LEVEL_DATA = False

MODEL = "OLS"
MODEL_VERSION = "ols_v1"
FEATURE_SET_ID = "microP_l1Imb_9Lags"
NORMALIZATION_ID = ""

In [ ]:
# ============================================================
# Initialize Run
# ============================================================
warnings.filterwarnings("ignore")

PARENT = os.path.dirname(os.getcwd())
DATA_ROOT = f"{PARENT}/data/processed"

# Per model family: model_outputs/Regression holds run_registry.csv plus
# DailyDiagnostics/<run_id> and TickLevel/<run_id>.
OUTPUT_ROOT = f"{PARENT}/model_outputs/Regression"

t = pd.read_parquet(f"{DATA_ROOT}/{du.SYMBOLS[0]}/{du.SAMPLE_DATES[0]}.parquet")

RUN_ID = mu.initialize_run(
    output_root=OUTPUT_ROOT,
    model_version=MODEL_VERSION,
    feature_set_id=FEATURE_SET_ID,
    normalization_id=NORMALIZATION_ID
)

DIRS = mu.create_output_dirs(
    output_root=OUTPUT_ROOT,
    run_id=RUN_ID,
)

TICK_OUTPUT_DIR = DIRS["tick"]
DAILY_OUTPUT_DIR = DIRS["daily"]

daily_results = []

# Trading days used for the rolling walk-forward
SD = du.SAMPLE_DATES[:]

sample_cols = pd.read_parquet(f"{DATA_ROOT}/{du.SYMBOLS[0]}/{SD[0]}.parquet").columns

FEATURE_COLS = list(sample_cols[sample_cols.str.startswith("F_")])
TARGET_COLS = list(sample_cols[sample_cols.str.startswith("T_")])

# ============================================================
# Main Loop
# ============================================================

GLOBAL_START = time.perf_counter()

# One symbol at a time: load all its days, then roll train(i) -> test(i+1).
# No feature/target normalization: OLS is scale-equivariant, so it does not
# affect fits or predictions. Revisit for regularized / NN models.
for symbol in tqdm(du.SYMBOLS[:-1], desc="Processing symbols", unit="symbol"):
    day_cache = mu.load_day_cache(DATA_ROOT, symbol, SD, FEATURE_COLS, TARGET_COLS)

    # --------------------------------------------------------
    # Rolling train/test
    # --------------------------------------------------------

    for i in tqdm(range(len(SD) - 1), desc=f"{symbol}", leave=False, unit="split"):

        train_day = SD[i]
        test_day = SD[i+1]

        train_cache = day_cache[train_day]
        test_cache = day_cache[test_day]

        X_train = train_cache["X"]
        X_test = test_cache["X"]

        Y_train = train_cache["Y"]
        Y_test = test_cache["Y"]

        model = LinearRegression()
        model.fit(X_train, Y_train)

        Y_pred = model.predict(X_test).astype(np.float32)

        # Residuals
        RESID = (Y_test - Y_pred).astype(np.float32)

        if SAVE_TICK_LEVEL_DATA:
            mu.save_tick_residuals(
                resid=RESID,
                timestamps=test_cache["timestamp"],
                target_cols=TARGET_COLS,
                tick_output_dir=TICK_OUTPUT_DIR,
                test_day=test_day,
                symbol=symbol
            )

        daily_results += mu.daily_diagnostic_rows(
            resid=RESID,
            Y_test=Y_test,
            target_cols=TARGET_COLS,
            train_day=train_day,
            test_day=test_day,
            symbol=symbol,
            run_id=RUN_ID,
            n_train=X_train.shape[0],
            n_test=X_test.shape[0]
        )

tqdm.write(f"\nTOTAL PIPELINE TIME: {time.perf_counter()-GLOBAL_START:.2f}s")
# ============================================================
# Save Outputs
# ============================================================

mu.save_table(
    df=pd.DataFrame(daily_results),
    root_dir=DAILY_OUTPUT_DIR,
    filename="daily_diagnostics.parquet",
)

In [5]:
daily_out = pd.read_parquet(f"{DAILY_OUTPUT_DIR}/daily_diagnostics.parquet")
daily_out[daily_out["mse_ratio"] < 1]

,train_day,test_day,symbol,target,run_id,n_train,n_test,mse,mse_bench,mse_ratio,mae,r2_oos
6,2023-01-02,2023-01-03,Adidas,T_TransactionPrice_LogReturn_1m,9,41890,113082,8.182859e-07,8.087230e-07,0.988313,0.000688,-0.011825
7,2023-01-02,2023-01-03,Adidas,T_TransactionPrice_LogReturn_2.5m,9,41890,113082,1.841167e-06,1.729504e-06,0.939352,0.001050,-0.064564
14,2023-01-02,2023-01-03,Adidas,T_MidPrice_LogReturn_30s,9,41890,113082,3.888250e-07,3.859200e-07,0.992529,0.000460,-0.007527
15,2023-01-02,2023-01-03,Adidas,T_MidPrice_LogReturn_1m,9,41890,113082,8.222920e-07,7.924667e-07,0.963729,0.000683,-0.037636
16,2023-01-02,2023-01-03,Adidas,T_MidPrice_LogReturn_2.5m,9,41890,113082,1.916357e-06,1.758951e-06,0.917861,0.001052,-0.089489
...,...,...,...,...,...,...,...,...,...,...,...,...
113394,2023-06-29,2023-06-30,Airbus,T_MicroPrice_LogReturn_10s,9,49321,48117,2.875172e-08,2.864654e-08,0.996342,0.000117,-0.003672
113395,2023-06-29,2023-06-30,Airbus,T_MicroPrice_LogReturn_15s,9,49321,48117,4.144114e-08,4.129498e-08,0.996473,0.000144,-0.003539
113396,2023-06-29,2023-06-30,Airbus,T_MicroPrice_LogReturn_30s,9,49321,48117,7.775171e-08,7.717340e-08,0.992562,0.000205,-0.007494
113398,2023-06-29,2023-06-30,Airbus,T_MicroPrice_LogReturn_2.5m,9,49321,48117,3.160467e-07,3.103057e-07,0.981835,0.000425,-0.018501
